# E-commerce Medallion Architecture Assignment — Starter Notebook

Use the seven supplied CSV files to build a Bronze, Silver, and Gold solution in Databricks Free Edition.

Do not copy the completed solution until you have attempted each task. Write your code in the `TODO` cells and record observations in markdown cells.

## Section 1 — Configure the environment

**Task 1:** Import PySpark functions and Window.  
**Task 2:** Detect the current catalog.  
**Task 3:** Create `ecom_bronze`, `ecom_silver`, and `ecom_gold` schemas.  
**Task 4:** Create a managed volume named `ecommerce_training_files` in the `default` schema.

In [1]:
from pyspark.sql import functions as F, SparkSession
from pyspark.sql.classic import dataframe
from pyspark.sql.window import Window
import os

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if not IS_DATABRICKS:
    os.environ.setdefault("HADOOP_HOME", "C:\\hadoop")
    os.environ["PATH"] = os.environ.get("PATH", "") + ";" + os.path.join(os.environ["HADOOP_HOME"], "bin")
    spark: SparkSession =  (SparkSession.builder.appName("Ecommerce Assignment")
                            .master("local[*]")
                            .getOrCreate())

CATALOG: str = spark.catalog.currentCatalog()

for schema in ("ecom_bronze", "ecom_silver", "ecom_gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.default.ecommerce_training_files
""")

VOLUME_PATH = f"/Volumes/{CATALOG}/default/ecommerce_training_files"

print("Version:", spark.version)
print("Catalog:", CATALOG)
print("Volume path:", VOLUME_PATH)
print("Current Schema", spark.catalog.currentDatabase())

G:\Owen\Revature\Training\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'VOLUME'. SQLSTATE: 42601 (line 2, pos 11)

== SQL ==

    CREATE VOLUME IF NOT EXISTS spark_catalog.default.ecommerce_training_files
-----------^^^


## Section 2 — Bronze ingestion

Load every CSV with:

- header enabled
- schema inference disabled
- all source columns retained as strings
- `_source_file`
- `_ingested_at`
- `_raw_row_hash`

Create one Bronze Delta table per source file.

In [ ]:
from pyspark.sql import DataFrame

FILE_TO_ENTITY = {
    "customers.csv": "customers_raw",
    "products.csv": "products_raw",
    "orders.csv": "orders_raw",
    "order_items.csv": "order_items_raw",
    "payments.csv": "payments_raw",
    "shipments.csv": "shipments_raw",
    "returns.csv": "returns_raw",
}

def create_dataframe(file_name: str) -> DataFrame:
    """Create a DataFrame from a CSV file with all columns as strings."""
    raw_dataframe: DataFrame = spark.read.csv(f"{VOLUME_PATH}/input/{file_name}", header=True, inferSchema=False)
    raw_dataframe: DataFrame = raw_dataframe.withColumns({
        "_raw_row_hash": F.hash(*raw_dataframe.columns),
        "_source_file": F.lit(file_name),
        "_ingested_at": F.lit(F.current_timestamp()),
    })
    return raw_dataframe

raw_dataframes: dict[str, DataFrame] = {value: create_dataframe(key) for key, value in FILE_TO_ENTITY.items()}

for name, df in raw_dataframes.items():
    df: DataFrame = df
    (df.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(f"{CATALOG}.ecom_bronze.{name}"))

## Section 3 — Bronze profiling

For every Bronze table calculate:

1. row count
2. column count
3. duplicate raw-row hash count
4. null or blank count for every business key
5. five sample rows

In [ ]:
bronze_dataframes: dict[str, DataFrame] = {value.replace("_raw", "_bronze"): spark.table(f"{CATALOG}.ecom_bronze.{value}") for value in FILE_TO_ENTITY.values()}

for name, df in bronze_dataframes.items():
    columns: list[str] = df.columns
    print(name)
    print("Row Count:", df.count())
    print("Column Count:", len(columns))
    print("Duplicate hashes", df.groupBy("_raw_row_hash").count().filter("count > 1").count())
    print("null/na", df.select([F.count(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), c)).alias(c) for c in columns]).show())
    print("Sample Rows:", df.show(5))

## Section 4 — Silver customers

Implement:

- trim and case normalization
- email validation
- multi-format date parsing
- gender and Boolean standardization
- loyalty-tier validation
- deduplication by `customer_id`
- valid table and quarantine table

In [ ]:
from pyspark.sql import Column
from pyspark.sql.window import Window, WindowSpec


def validate(validator: Column, invalid_reason: str) -> dict[str, Column]:
    """
    Takes a validator column (a True/False evaluated column), and sets the record to invalid if validator is False and adds the invalid
    :param validator: Column to validate
    :param invalid_reason: Reason for invalidation
    :return: A dictionary with updated validation columns
    :param invalid_reason:
    :return: A dictionary with updated validation columns

    """
    is_valid: Column = F.when(validator, F.lit(True)).otherwise(F.lit(False))
    return {
        "is_valid_record": F.when(~F.col("is_valid_record"), F.lit(False)).otherwise(is_valid),
        "validation_error_count": F.when(is_valid, F.col("validation_error_count")).otherwise(F.col("validation_error_count") + 1),
        "validation_errors": F.when(is_valid, F.col("validation_errors")).otherwise(F.concat_ws("|", F.col("validation_errors"), F.lit(invalid_reason)))
    }
# TODO: create silver customers and customers_quarantine.

def normalize_str(column_name: str) -> Column:
    return F.trim(F.lower(F.initcap(F.col(column_name))))

def normalize_phone(column_name: str) -> Column:
    return F.trim(F.replace(F.col(column_name), r"-\(\)", ""))

def normalize_gender(column_name: str) -> Column:
    male: list[str] = ["M", "MALE"]
    female: list[str] = ["F", "FEMALE"]
    return F.when(F.upper(F.col(column_name)).isin(male), "M").when(F.upper(F.col(column_name)).isin(female), "F").otherwise("O")

def normalize_bool(column_name: str) -> Column:
    return F.when(F.upper(F.col(column_name)).isin(["TRUE", "T", "YES", "Y", "1"]), True).otherwise(False)

silver_customers: DataFrame = bronze_dataframes["customers_bronze"]

#data normalization

silver_customers = silver_customers.withColumns({
    "first_name": normalize_str("first_name"),
    "last_name": normalize_str("last_name"),
    "phone": normalize_phone("phone"),
    "date_of_birth": F.try_to_date(F.col("date_of_birth")),
    "gender": normalize_gender("gender"),
    "address": normalize_str("address"),
    "city": normalize_str("city"),
    "state": normalize_str("state"),
    "signup_date": F.try_to_date(F.col("signup_date")),
    "loyalty_tier": normalize_str("loyalty_tier"),
    "is_active": normalize_bool("is_active"),
})

#validation
silver_customers = silver_customers.withColumns({
    "is_valid_record": F.lit(True),
    "validation_error_count": F.lit(0),
    "validation_errors": F.lit(""),
})

silver_customers = silver_customers.withColumns(
    validate(F.regexp(F.col("phone"), F.lit(r'^\d{10}$')), "INVALID_PHONE")
)

silver_customers = silver_customers.withColumns(validate(F.col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), "INVALID_EMAIL"))

silver_customers = silver_customers.withColumns(
    validate(F.col("loyalty_tier").isin(["Bronze", "Silver", "Gold", "Platinum"]), "INVALID_LOYALTY_TIER")
)


def de_duplicate(df: DataFrame, id_column: str) -> DataFrame:
    df = df.withColumn("_dedup_row_id", F.monotonically_increasing_id())
    exact_duplicate_window: WindowSpec = (Window
                                   .partitionBy("_raw_row_hash")
                                   .orderBy(F.col("_ingested_at").asc(),
                                            F.col("_raw_row_hash").asc(),
                                            F.col("_dedup_row_id").asc()))
    id_column_window: WindowSpec = (Window
                                   .partitionBy(id_column)
                                   .orderBy(F.col("_ingested_at").asc(),
                                            F.col("_raw_row_hash").asc(),
                                            F.col("_dedup_row_id").asc()))

    df = df.withColumns({
    "_exact_duplicate_rank": F.row_number().over(exact_duplicate_window),
    "_id_column_rank": F.row_number().over(id_column_window),
    })

    # Exact duplicates: content-identical rows (same _raw_row_hash). Keep rank 1, reject the rest.
    df = df.withColumns(
        validate(F.col("_exact_duplicate_rank") == 1, "EXACT_DUPLICATE"))

    # Duplicate customer_id (content may differ): keep rank 1 per the ordering rule above, reject the rest.
    df = df.withColumns(
        validate(F.col("_id_column_rank") == 1, "DUPLICATE_CUSTOMER_ID"))

    # drop helper columns used only for ranking
    df = df.drop(
        "_dedup_row_id", "_exact_duplicate_rank", "_id_column_rank"
    )
    return df

silver_customers = de_duplicate(silver_customers, "customer_id")

customers_quarantine = silver_customers.filter(~F.col("is_valid_record"))

silver_customers = silver_customers.filter(F.col("is_valid_record"))


## Section 5 — Silver products

Validate product ID, category, price, cost, stock, rating, created date, active flag, and duplicate IDs.

In [ ]:
silver_products: DataFrame = bronze_dataframes["products_bronze"]

silver_products = silver_products.withColumns({
    "product_name": normalize_str("product_name"),
    "category": normalize_str("category"),
    "subcategory": normalize_str("subcategory"),
    "brand": normalize_str("brand"),
    "unit_price": F.col("unit_price").cast("decimal(18,2)"),
    "cost_price": F.col("cost_price").cast("decimal(18,2)"),
    "stock_quantity": F.col("stock_quantity").cast("int"),
    "product_rating": F.col("product_rating").cast("decimal(4,1)"),
    "created_date": F.try_to_date(F.col("created_date")),
    "is_active": normalize_bool("is_active"),
})

silver_products = de_duplicate(silver_products, "product_id")

products_quarantine = silver_products.filter(~F.col("is_valid_record"))

silver_products = silver_products.filter(F.col("is_valid_record"))


## Section 6 — Silver orders

Implement:

- timestamp parsing
- order-status and channel normalization
- decimal type casting
- total formula check
- customer referential integrity
- duplicate-order handling

In [ ]:
# TODO: create silver orders and orders_quarantine.

silver_orders: DataFrame = bronze_dataframes["orders_bronze"]

silver_orders = silver_orders.withColumns({
    "order_date": F.try_to_timestamp(F.col("order_date")),
    "order_status": normalize_str("order_status"),
    "channel": normalize_str("channel"),
    "currency": F.upper(F.col("currency")),
    "subtotal": F.col("subtotal").cast("decimal(18,2)"),
    "discount_amount": F.col("discount_amount").cast("decimal(18,2)"),
    "tax_amount": F.col("tax_amount").cast("decimal(18,2)"),
    "shipping_cost": F.col("shipping_cost").cast("decimal(18,2)"),
    "total_amount": F.col("total_amount").cast("decimal(18,2)"),
    "shipping_city": normalize_str("shipping_city"),
    "shipping_state": normalize_str("shipping_state"),
    "shipping_country": normalize_str("shipping_country"),
    "last_updated": F.try_to_timestamp(F.col("last_updated")),
})

total_correct: Column = F.col("total_amount") == (F.col("subtotal") - F.col("discount_amount") + F.col("tax_amount") + F.col("shipping_cost"))

silver_orders = silver_orders.withColumns(
    validate(total_correct, "INVALID_TOTAL")
)

def valid_fk(df: DataFrame, fk_column: str, ref_df: DataFrame, ref_column: str, invalid_reason: str) -> DataFrame:
    """
    Validates a foreign key by left-joining df against the distinct keys in ref_df.
    Rows whose fk_column has no match in ref_df (null after the join) are marked invalid.
    :param df: DataFrame containing the foreign key
    :param fk_column: Name of the foreign key column in df
    :param ref_df: Reference DataFrame containing the valid keys
    :param ref_column: Name of the reference column in ref_df
    :param invalid_reason: Reason recorded on rows that fail validation
    :return: The validated DataFrame, with the join helper column removed
    """
    ref_keys: DataFrame = ref_df.select(F.col(ref_column).alias("_fk_ref_key")).distinct()
    df = df.join(ref_keys, df[fk_column] == ref_keys["_fk_ref_key"], "left")
    df = df.withColumns(validate(F.col("_fk_ref_key").isNotNull(), invalid_reason))
    return df.drop("_fk_ref_key")

silver_orders = valid_fk(silver_orders, "customer_id", silver_customers, "customer_id", "INVALID_CUSTOMER_ID")

silver_orders = de_duplicate(silver_orders, "order_id")

orders_quarantine = silver_orders.filter(~F.col("is_valid_record"))

silver_orders = silver_orders.filter(F.col("is_valid_record"))


## Section 7 — Silver order items

Implement quantity, unit-price, discount, line-total, order reference, product reference, and duplicate-item checks.

In [ ]:
# TODO: create silver order_items and order_items_quarantine.

## Section 8 — Silver payments, shipments, and returns

In [ ]:
# TODO: create valid and quarantine tables for payments.

In [ ]:
# TODO: create valid and quarantine tables for shipments.

In [ ]:
# TODO: create valid and quarantine tables for returns.

## Section 9 — Spark SQL practice

Create temporary views for all valid Silver tables and solve the SQL questions from `Assignment_Questions.md`.

In [ ]:
# TODO: register Silver tables as temporary views.

In [ ]:
%sql
-- TODO 1: sales by order status

In [ ]:
%sql
-- TODO 2: top 20 customers by lifetime value

In [ ]:
%sql
-- TODO 3: category revenue rank using a CTE and window function

In [ ]:
%sql
-- TODO 4: daily revenue with lag and running total

## Section 10 — Gold layer

Create:

- `dim_customer`
- `dim_product`
- `fact_order_line`
- `daily_sales_kpi`
- `category_performance`
- `customer_360`

In [ ]:
# TODO: create Gold dimensions.

In [ ]:
# TODO: create Gold fact table using Spark SQL.

In [ ]:
# TODO: create Gold KPI tables.

## Section 11 — Delta Lake DML

Create a safe product-copy table and practice:

1. `MERGE`
2. `UPDATE`
3. `DELETE`
4. `DESCRIBE HISTORY`

In [ ]:
# TODO: implement the Delta DML exercise without changing the main Silver table.

## Section 12 — Validation

Reconcile every Bronze count with valid plus quarantined Silver counts. Confirm no duplicate business keys and no orphan rows remain in valid Silver tables.

In [ ]:
# TODO: produce a final quality dashboard.